#### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [6]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing olde context. Summarization is useful for the following:
Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# Message based summarization using SummarizationMiddleware

agent = create_agent(model="groq:llama-3.1-8b-instant", checkpointer=InMemorySaver(),
        middleware=[SummarizationMiddleware(
            model = "groq:llama-3.1-8b-instant",
            trigger = ("messages", 10),
            keep = ("messages",5)
        )])



In [19]:
# Run with thread id

config = {"configurable":{"thread_id":"test_1"}}

#### Summarization based on token size

In [20]:
questions = [
    "what is the capital of France?",
    "what is the capital of Germany?",
    "what is the capital of Italy?",
    "what is the capital of Spain?",
    "what is the capital of Portugal?",
    "what is the capital of Netherlands?",
    "what is the capital of Belgium?",
    "what is the capital of Switzerland?",
    "what is the capital of Austria?",
    "what is the capital of Poland?",
    "what is the capital of Czech Republic?",
    "what is the capital of Hungary?",
    "what is the capital of Slovakia?"
]

for question in questions:
    response = agent.invoke({"messages":[HumanMessage(content=question)]}, config)
    print(f"Message: {question}")
    print(f"Response: {response}")
    print(f"Messages: {len(response['messages'])}")

Message: what is the capital of France?
Response: {'messages': [HumanMessage(content='what is the capital of France?', additional_kwargs={}, response_metadata={}, id='057007c9-2358-422d-a2a6-a06cb40778e3'), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.006596131, 'completion_tokens_details': None, 'prompt_time': 0.002042196, 'prompt_tokens_details': None, 'queue_time': 0.054608803, 'total_time': 0.008638327}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa454-7ac3-7fd1-af9a-547608ec7fde-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}
Messages: 2
Message: what is the capital of Germany?
Response: {'messages': [HumanMessage(c

Summarization based on Token size

In [34]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def find_hotels_in_city(city: str):
    """Find hotels in the given city."""
    return f"Hotels in {city}: Hotel A, Hotel B, Hotel C"



Why is thread_id needed?
When you use a checkpointer like InMemorySaver, LangGraph needs a way to know which conversation's state it should load and save.

| Thread ID | Conversation                     |
| --------- | -------------------------------- |
| `test_1`  | User A's chat                    |
| `test_2`  | User B's chat                    |
| `abc123`  | Another independent conversation |


In [29]:
config = {"configurable":{"thread_id":"test_2"}}

In [31]:
#Token counter (approximate)
def count_tokens (messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token


In [39]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[find_hotels_in_city],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("tokens", 500),
            keep=("tokens", 200),
        )
    ],
)

In [ ]:
cities = [
    "Paris",
    "London",
    "Tokyo",
    "New York",
    "Dubai",
    "Singapore",
    "Sydney",
    "Rome",
    "Berlin",
    "Amsterdam",
    "Barcelona",
    "Bangkok",
]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"What are the tourist spots in {city}?"
                )
            ]
        },
        config=config,
    )

    tokens = count_tokens(response["messages"])

    print("=" * 1000)
    print(f"City: {city}")
    print(f"Approx Tokens: {tokens}")
    print(f"Total Messages: {len(response['messages'])}")
    print(response["messages"][-1].content)